# 새 M2·M5 — seed 43 / 최대 300 epoch / 공통 조기 종료

## 이번에 실행하는 것
**새 M2와 새 M5 두 개**만 우선 학습합니다. 추가 시드와 상수-q 대조군은 자동 실행하지 않습니다.
M1·기존 M2의 호환 학습곡선은 조기 종료 규칙을 시간순으로 재현해 재사용합니다. M4는 **보완 M4**로 고정합니다.

## 구조와 고정 조건
- N/V: h_N=W_N[p_N;q_N]+b_N, h_V=W_V[p_V;q_V]+b_V. 총 48개 추가 파라미터.
- N/V를 모두 유지하는 선형 결합이며 비선형 게이트가 아닙니다. M2에는 q_C가 없습니다.
- ID LightGCN과 하나의 optimizer·추천손실로 공동학습합니다. 동결·외부 재정렬은 없습니다.
- 학습은 양성 상품 leave-one-out, 평가는 전체 학습이력. 유효하지 않거나 남은 이력이 없으면 변환 후에도 0입니다.
- 변환 출력은 단위정규화하지 않습니다. 영향력·노름을 측정하며 scale 문제가 해결됐다고 미리 주장하지 않습니다.
- Dunnhumby 개발 684–690, seed 43, rho=0.05, ID 64차원, N/V 축 4차원, 2층, L2=1e-3, lr=5e-4, batch=8192, K=1 uniform, binary graph, MIN_ITEM_INTER=1.
- M5는 보완 가중식 1+0.5*q_C*(1-RBF_value_fit)을 사용합니다. 원형 M4로 바꾸지 않습니다.

## 공통 선택 규칙
- 최대 300 epoch, **25 epoch마다 개발평가**.
- 100 epoch까지 평가·최고값을 기록하되 미개선 대기는 세지 않습니다. 100에서 대기 횟수를 0으로 초기화합니다.
- 이후 **4회 연속 평가에서 최고값이 개선되지 않으면 종료**합니다. 예: 125·150·175·200 미개선이면 200 종료.
- 선택 지표: **전체 가격·구매금액 가중 적중값@10**. min_delta=0, 엄격한 증가만 개선, 동률은 이른 checkpoint.
- 종료 시 관찰한 평가 지점 전체에서 가장 좋은 checkpoint로 복원합니다. 모델마다 선택·종료 epoch가 달라도 됩니다.
- 다른 지표의 최고 epoch를 골라 한 표에 섞지 않습니다. 조기 종료가 늦은 개선을 항상 찾아주지는 않습니다.
- **최종 test·holdout 미사용**. 반복 노출된 개발분할의 선택 결과이며 유의성·안정성·일반화·CLV 귀속 주장이 아닙니다.

## 판정과 비용
M5의 전체 가중 적중값@10 및 가중 NDCG@10이 선택된 M4보다 모두 높은지, 전체 Recall/NDCG @10·20·50이 각각 선택된 M1의 99% 이상인지 봅니다.
M4만 이겼는지와 M1까지 넘어서 전체 목표에 가까워졌는지는 별도로 표시합니다. @20/@50·전 CLV 구간·노출도 생략 없이 저장합니다.

**새 2개 학습은 최대 600 model-epoch**입니다. 여기에 호환 결과가 없는 M1/M4의 추가 학습이 필요할 수 있습니다. 실제 시간은 GPU·평가·저장에 따라 다릅니다.
100 epoch 최종 결과만 있는 M4는 조기 종료에 필요한 앞선 평가곡선을 복원할 수 없어 단순 연장 대상으로 쓰지 않습니다.
준비 셀에서 추가 학습 대상과 최대 epoch 수를 확인하고, 허용 목록에 직접 적어야 시작됩니다. 기존 M2 결과가 없으면 비교에서 빠지며 자동 재학습하지 않습니다.

## 실행 안내
1–3번 셀은 준비·계획 확인이며 학습하지 않습니다. **4번 셀부터 학습**합니다.
진행 중인 H&M 실험과 별도 런타임에서 실행하세요. 매 epoch optimizer·난수·대기 상태를 저장하며 같은 설정으로 재실행하면 이어집니다. Colab 세션 자체를 자동 재연결하지는 않습니다.
이 노트북은 이전 3시드·100 epoch 버전을 대체합니다. 이전 결과·러너 파일은 보존합니다.


In [ ]:
# 1. 기존 실험과 분리된 고정 코드
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json
from pathlib import Path
SOURCE_COMMIT = 'd3b523321e948f0e7654b61681d1f3adc6b50668'
REPO = Path('/content/clv-history-linear-nv-es-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '이전 모듈이 남아 있습니다. 별도 런타임 또는 세션 재시작이 필요합니다.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('고정 실행 코드:', SOURCE_COMMIT)


In [ ]:
# 2. 설정 출력만 — 기본은 seed 43, 상수 대조 미실행
import torch
import pandas as pd
import lightgcn_clv_history_linear_nv_early_stop as screen
cfg = screen.configure()
print(json.dumps(screen.preflight(cfg), ensure_ascii=False, indent=2))
print('결과 폴더:', cfg.out_dir)
assert torch.cuda.is_available(), '학습을 위해 GPU 런타임을 선택하세요.'


In [ ]:
# 3. 기존 곡선 재사용 + 실행 비용 계획 — 학습 없음
prepared = screen.prepare(cfg)
plan = pd.DataFrame(prepared['plan'])
display(plan)
anchors = prepared['anchors']
if anchors:
    display(pd.DataFrame([{k: a.get(k) for k in (
        'seed', 'model_id', 'origin', 'selected_epoch', 'stopped_epoch', 'source_result')} for a in anchors]))
print('최대 추가 model-epoch 합계:', int(plan.max_additional_epochs.sum()))
required = [f"{p['seed']}:{p['model_id']}" for p in prepared['plan']
            if p['model_id'] in ('m1', 'm4') and p['action'] == 'train_from_start']
print('추가 기준모형 학습 승인이 필요한 항목:', required)
print('이 목록을 확인한 뒤 필요한 항목만 다음 셀의 APPROVED_BASELINE_FITS에 적으세요.')


In [ ]:
# 4. 실제 학습 시작
# 예: 계획표에 43:m4만 추가 학습이 필요할 때 ['43:m4']로 입력합니다.
# 빈 목록이 기본입니다. 필요한 승인이 빠지면 새 M2/M5도 학습 전에 중단됩니다.
APPROVED_BASELINE_FITS = []
absolute = screen.run(cfg, prepared=prepared, approved_baseline_fits=APPROVED_BASELINE_FITS)
print(json.dumps(absolute.attrs['paths'], ensure_ascii=False, indent=2))


In [ ]:
# 5. 선택된 단일 checkpoint의 전체 지표 비교
paths = absolute.attrs['paths']
summary = pd.read_csv(paths['summary'])
reading = pd.read_csv(paths['reading'])
display(absolute[['seed','model_id','selected_epoch','stopped_epoch','stop_reason']])
key_metrics = list(screen.base.ACCURACY) + list(screen.fixed.PRIMARY) + [
    'price_purchase_amount_weighted_hit@20', 'price_purchase_amount_weighted_hit@50',
    'vndcg@20', 'vndcg@50']
display(summary[summary.metric.isin(key_metrics)].reset_index(drop=True))
display(reading)
print('directional_pass: M2-M1 또는 M5-M4 개선 + M1 정확도 보호')
print('overall_goal_direction: 위 조건에 더해 M1 대비 경제지표도 개선')
print('단일 개발 seed의 방향 점검입니다. 통과해도 성공·유의성 확정이 아닙니다.')
for name, path in paths.items():
    print(name, ':', path)


In [ ]:
# 6. 전체 결과 ZIP 다운로드 (체크포인트는 큰 파일이므로 제외)
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path('/content/m2_m5_linear_nv_results.zip')
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as z:
    for path in paths.values():
        z.write(path, arcname=Path(path).name)
files.download(str(archive))
